In [ ]:
import pandas as pd
import os
import numpy as np
from ast import literal_eval
import matplotlib.pyplot as plt
import datetime

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
def recommend_products(conditions, price_min, price_max, df, preferred_types=None):
    condition_to_ingredients_weighted = {
    "Acne": {
        "salicylic acid": 3,
        "benzoyl peroxide": 3,
        "retinoids": 3,
        "niacinamide": 3,
        "azelaic acid": 3,
        "adapalene": 1,
        "alpha-hydroxy acids": 1,
        "sulfur": 1,
        "tea tree oil": 1,
        "succinic acid": 1
    },
    "Dry Skin": {
        "hyaluronic acid": 3,
        "ceramides": 3,
        "niacinamide": 3,
        "glycerin": 3,
        "shea butter": 3,
        "squalane": 1,
        "vitamin e": 1,
        "dimethicone": 1,
        "petrolatum": 1
    },
    "Oily Skin": {
        "salicylic acid": 3,
        "niacinamide": 3,
        "clay": 3,
        "hyaluronic acid": 3,
        "retinol": 3,
        "tea tree oil": 1,
        "lactic acid": 1,
        "astringents": 1
    },
    "Dark Spots": {
        "hydroquinone": 3,
        "vitamin c": 3,
        "kojic acid": 3,
        "retinol": 3,
        "niacinamide": 3,
        "glycolic acid": 1,
        "azelaic acid": 1,
        "tranexamic acid": 1
    },
    "Wrinkles": {
        "hyaluronic acid": 3,
        "collagen": 3,
        "retinol": 3,
        "vitamin c": 3,
        "niacinamide": 3,
        "glycolic acid": 1,
        "bakuchiol": 1,
        "ferulic acid": 1,
        "peptides": 1,
        "polyglutamic acid": 1
    },
    "Eye bags": {
        "hyaluronic acid": 3,
        "adipoless": 3,
        "peptides": 3,
        "sabiwhite": 3,
        "syn-eye": 3,
        "caffeine": 1,
        "vitamin c": 1
    },
    "Pores": {
        "niacinamide": 3,
        "green tea": 3,
        "azelaic acid": 3,
        "salicylic acid": 3,
        "retinol": 3,
        "alpha-hydroxy acids": 1,
        "charcoal": 1,
        "kaolin clay": 1
    },
    "Normal Skin": {
        "ceramides": 3,
        "hyaluronic acid": 3,
        "niacinamide": 3,
        "squalane": 3,
        "lactic acid": 3
    },
    "Combination Skin": {
        "superoxide dismutase": 3,
        "lactic acid": 3,
        "hyaluronic acid": 3,
        "peptides": 3,
        "squalane": 3,
        "alpha-hydroxy acids": 1,
        "green clay": 1,
        "salicylic acid": 1,
        "glycolic acid": 1,
        "witch hazel": 1
    }
}

    def score_product_weighted(row):
        try:
            ingreds = [i.lower() for i in eval(row['clean_ingreds'])]
        except:
            ingreds = []

        score = 0
        for cond in conditions:
            weights = condition_to_ingredients_weighted.get(cond, {})
            for ing in ingreds:
                score += weights.get(ing, 0)

        type_bonus = 1 if preferred_types and row['product_type'] in preferred_types else 0
        return score + type_bonus

    df_filtered = df[
        (df['price_clean'] >= price_min) & (df['price_clean'] <= price_max)
    ].copy()

    df_filtered['match_score'] = df_filtered.apply(score_product_weighted, axis=1)
    results = df_filtered[df_filtered['match_score'] > 0]
    results = results.sort_values(by='match_score', ascending=False)

    return results.head(5).to_dict(orient='records')

In [ ]:
import os
os.listdir("/content/drive/MyDrive/DermaVue Project")

In [ ]:
project_drive_path = "/content/drive/MyDrive/DermaVue Project"
product_path = os.path.join(project_drive_path, "skincare_products_clean.csv")
df_products = pd.read_csv(product_path)
df_products['price_clean'] = (
    df_products['price']
    .replace('[£€,]', '', regex=True)
    .str.strip()
    .astype(float))

In [ ]:
def precision_at_k(test_conditions, k=5):
    scores = []

    for conditions in test_conditions:
        recs = recommend_products(conditions, 5, 50, df_products)

        target_ingredients = set()
        for c in conditions:
            target_ingredients.update(condition_to_ingredients.get(c, []))

        relevant = 0
        for r in recs[:k]:
            try:
                ingreds = [i.lower() for i in literal_eval(r['clean_ingreds'])]
                if any(ing in ingreds for ing in target_ingredients):
                    relevant += 1
            except:
                continue

        scores.append(relevant / k)

    return np.mean(scores)

In [ ]:
def coverage(condition_list):
    covered = 0
    for cond in condition_list:
        recs = recommend_products([cond], 5, 50, df_products)
        if len(recs) > 0:
            covered += 1
    return covered / len(condition_list)

In [ ]:
def plot_match_score_distribution(conditions, save_path=None):
    recs = recommend_products(conditions, 5, 50, df_products)
    scores = [r['match_score'] for r in recs]

    plt.figure(figsize=(6, 4))
    plt.hist(scores, bins=range(0, max(scores)+2), align='left', edgecolor='black')
    plt.title(f"Match Score Distribution for {', '.join(conditions)}")
    plt.xlabel("Match Score")
    plt.ylabel("Number of Products")
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path)
    else:
        plt.show()

    return scores

In [ ]:
condition_to_ingredients_weighted = {
    "Acne": {
        "salicylic acid": 3,
        "benzoyl peroxide": 3,
        "retinoids": 3,
        "niacinamide": 3,
        "azelaic acid": 3,
        "adapalene": 1,
        "alpha-hydroxy acids": 1,
        "sulfur": 1,
        "tea tree oil": 1,
        "succinic acid": 1
    },
    "Dry Skin": {
        "hyaluronic acid": 3,
        "ceramides": 3,
        "niacinamide": 3,
        "glycerin": 3,
        "shea butter": 3,
        "squalane": 1,
        "vitamin e": 1,
        "dimethicone": 1,
        "petrolatum": 1
    },
    "Oily Skin": {
        "salicylic acid": 3,
        "niacinamide": 3,
        "clay": 3,
        "hyaluronic acid": 3,
        "retinol": 3,
        "tea tree oil": 1,
        "lactic acid": 1,
        "astringents": 1
    },
    "Dark Spots": {
        "hydroquinone": 3,
        "vitamin c": 3,
        "kojic acid": 3,
        "retinol": 3,
        "niacinamide": 3,
        "glycolic acid": 1,
        "azelaic acid": 1,
        "tranexamic acid": 1
    },
    "Wrinkles": {
        "hyaluronic acid": 3,
        "collagen": 3,
        "retinol": 3,
        "vitamin c": 3,
        "niacinamide": 3,
        "glycolic acid": 1,
        "bakuchiol": 1,
        "ferulic acid": 1,
        "peptides": 1,
        "polyglutamic acid": 1
    },
    "Eye bags": {
        "hyaluronic acid": 3,
        "adipoless": 3,
        "peptides": 3,
        "sabiwhite": 3,
        "syn-eye": 3,
        "caffeine": 1,
        "vitamin c": 1
    },
    "Pores": {
        "niacinamide": 3,
        "green tea": 3,
        "azelaic acid": 3,
        "salicylic acid": 3,
        "retinol": 3,
        "alpha-hydroxy acids": 1,
        "charcoal": 1,
        "kaolin clay": 1
    },
    "Normal Skin": {
        "ceramides": 3,
        "hyaluronic acid": 3,
        "niacinamide": 3,
        "squalane": 3,
        "lactic acid": 3
    },
    "Combination Skin": {
        "superoxide dismutase": 3,
        "lactic acid": 3,
        "hyaluronic acid": 3,
        "peptides": 3,
        "squalane": 3,
        "alpha-hydroxy acids": 1,
        "green clay": 1,
        "salicylic acid": 1,
        "glycolic acid": 1,
        "witch hazel": 1
    }
}

In [ ]:
all_conditions = [
    ["Acne"],
    ["Dry Skin"],
    ["Oily Skin"],
    ["Dark Spots"],
    ["Wrinkles"],
    ["Eye bags"],
    ["Pores"],
    ["Normal Skin"],
    ["Combination Skin"],
]

for cond in all_conditions:
    scores = plot_match_score_distribution(cond)
    print(f"Mean score: {np.mean(scores):.2f}, Max score: {np.max(scores)}")

In [ ]:
import pandas as pd
import numpy as np

from ast import literal_eval

def get_summary_metrics(condition_list, df_products, k=5):
    summary = []

    for cond in condition_list:
        recs = recommend_products(cond, 5, 50, df_products)
        condition_set = set(cond)

        full_ingredient_weights = {}
        for c in cond:
            full_ingredient_weights.update(condition_to_ingredients_weighted.get(c, {}))
        target_ingredients = set(full_ingredient_weights.keys())

        if recs:
            scores = [r["match_score"] for r in recs]

            relevant = 0
            found_ingredients = set()
            for r in recs[:k]:
                try:
                    ingreds = [i.lower() for i in literal_eval(r["clean_ingreds"])]
                    if any(ing in target_ingredients for ing in ingreds):
                        relevant += 1
                        found_ingredients.update([ing for ing in ingreds if ing in target_ingredients])
                except:
                    continue

            precision = relevant / k
            recall = len(found_ingredients) / len(target_ingredients) if target_ingredients else 0
            coverage = 1

            summary.append({
                "Condition": ", ".join(cond),
                "Mean Score": np.mean(scores),
                "Max Score": np.max(scores),
                "Min Score": np.min(scores),
                "Std Dev": np.std(scores),
                "Num Results": len(scores),
                "Precision@5": round(precision, 2),
                "Recall": round(recall, 2),
                "Coverage": coverage
            })
        else:
            summary.append({
                "Condition": ", ".join(cond),
                "Mean Score": 0,
                "Max Score": 0,
                "Min Score": 0,
                "Std Dev": 0,
                "Num Results": 0,
                "Precision@5": 0,
                "Recall": 0,
                "Coverage": 0
            })

    return pd.DataFrame(summary)


score_df = get_summary_metrics(all_conditions, df_products)
score_df
